# Study 828 — FX Dollar Factor 💵

**Is the "dollar factor" a priced risk premium — and can the forward discount time it?**

Lustig, Roussanov & Verdelhan (2011) name two currency factors: the carry slope **HML_FX**
and the **dollar factor DOL** — the equal-weight average return of a basket of foreign
currencies against the USD. DOL's *unconditional* premium is famously small; their claim is
that it is priced **conditionally**, timed by the **average forward discount**. We build DOL on
a 7-currency G10 basket (2003-12-31 → 2026-06-30, 270 months) and test
both — carefully inverting the USD-base quotes so every pair reads USD-per-foreign.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership G10 — magnitudes are an upper bound.*


## 1. The idea in one picture

The **dollar factor** is just the average of every foreign currency's move against the USD. If you borrow dollars and hold a basket of foreign currencies, DOL is what you earn on the spot leg — positive when the dollar *weakens*. LRV say this level factor is priced, but only **conditionally**: it should pay more when the average **forward discount** (foreign rates minus US rates) is high. We test the premium and the timing.

In [1]:
R = dict(spot_bps=0.52, spot_ann=0.06, spot_t_nw=0.04, tim_t=-1.46, era_early_ann=1.07, era_late_ann=-0.9)
print('DOL premium (spot): %+.2f bps/mo = %+.2f%%/yr  (Newey-West t = %+.2f)'
      % (R['spot_bps'], R['spot_ann'], R['spot_t_nw']))
print('  -> a t of %+.2f is indistinguishable from zero' % R['spot_t_nw'])
print('dollar-timing slope t = %+.2f  (insignificant, and wrong sign)' % R['tim_t'])
print('era split: %+.2f%%/yr then %+.2f%%/yr -> flips sign'
      % (R['era_early_ann'], R['era_late_ann']))

DOL premium (spot): +0.52 bps/mo = +0.06%/yr  (Newey-West t = +0.04)
  -> a t of +0.04 is indistinguishable from zero
dollar-timing slope t = -1.46  (insignificant, and wrong sign)
era split: +1.07%/yr then -0.90%/yr -> flips sign


## 2. Is the engine honest? A live synthetic control

We plant a real dollar premium (`edge>0`) and a real timing relation (`timing>0`) in a seeded toy world and check the detector recovers each — and that it stays *silent* on the null (`edge=0, timing=0`). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from dollar_factor import data, strategy as st
null    = st.synthetic_detect(data.synthetic_panel(edge=0.0,   timing=0.0,  seed=828, n_months=480))
prem    = st.synthetic_detect(data.synthetic_panel(edge=0.004, timing=0.0,  seed=828, n_months=480))
timing  = st.synthetic_detect(data.synthetic_panel(edge=0.0,   timing=0.02, seed=828, n_months=480))
print('null world    : premium t = %+.2f, timing t = %+.2f  (both ~0)' % (null['prem_t_nw'], null['timing_t']))
print('planted premium: premium t = %+.2f  (should light up)' % prem['prem_t_nw'])
print('planted timing : timing  t = %+.2f  (should light up)' % timing['timing_t'])

null world    : premium t = +0.36, timing t = +1.70  (both ~0)
planted premium: premium t = +3.63  (should light up)
planted timing : timing  t = +5.23  (should light up)


## 3. The honest verdict — the dollar factor pays nothing here

On a 7-currency G10 basket the unconditional DOL premium is **+0.52 bps/mo = +0.06%/yr** with NW *t* = **+0.04** — a premium indistinguishable from zero, which is exactly LRV's own starting point. The **dollar-timing** test, run with the only conditioning variable we can build from yfinance spot (a trailing-dollar-trend proxy for the average forward discount), is **insignificant and wrong-signed** (NW *t* = **-1.46**, placebo p = 0.084), and the premium flips sign across eras. The synthetic control recovers *planted* versions of both cleanly, so this is a genuine absence, not a bug. **Signal: None** (the claimed edge is absent, with the true rate-based forward discount not reconstructable from spot — a data limit), **Tradability: Mirage** (both books lose money net).